In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from loaders._load_vn30_reg import preprocess, VN30, TARGETS
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_percentage_error

In [6]:
track = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=2)
    X_train, Y_train = data['train']
    X_val, Y_val = data['val']
    X_test, Y_test = data['test']
    target_scaler = data['scaler']['target']

    tscv = TimeSeriesSplit(n_splits=3)
    model = RandomizedSearchCV(
        estimator=DecisionTreeRegressor(),
        param_distributions={
            "max_depth": [3, 5, 7, 9],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }, 
        cv=tscv, 
        n_iter=10, 
        random_state=42
    )

    model.fit(X_train, Y_train)

    Y_pred = model.predict(X_test)
    Y_pred = target_scaler.inverse_transform(Y_pred)
    Y_test = target_scaler.inverse_transform(Y_test)

    r2 = r2_score(Y_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_test, Y_pred) * 100

    track["r2"].append(r2)
    track["mape"].append(mape)

    print(f"Symbol: {symbol}, R^2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R^2: {np.mean(track['r2']):.4f}, Mean MAPE: {np.mean(track['mape']):.4f}")
print(f"Std R^2: {np.std(track['r2']):.4f}, Std MAPE: {np.std(track['mape']):.4f}")

Symbol: ACB, R^2: -10.1420, MAPE: 16.7933
Symbol: BCM, R^2: 0.9415, MAPE: 1.7251
Symbol: BID, R^2: -4.3718, MAPE: 10.6184
Symbol: BVH, R^2: 0.9747, MAPE: 1.3237
Symbol: CTG, R^2: 0.5666, MAPE: 3.3002
Symbol: FPT, R^2: -3.5375, MAPE: 29.0799
Symbol: GAS, R^2: 0.9546, MAPE: 0.8204
Symbol: GVR, R^2: 0.9538, MAPE: 2.0120
Symbol: HDB, R^2: -3.3313, MAPE: 18.0551
Symbol: HPG, R^2: 0.6158, MAPE: 2.3877
Symbol: LPB, R^2: -2.1147, MAPE: 35.7083
Symbol: MBB, R^2: 0.2305, MAPE: 4.1269
Symbol: MSN, R^2: 0.9348, MAPE: 1.5376
Symbol: MWG, R^2: 0.9741, MAPE: 1.4630
Symbol: PLX, R^2: 0.9783, MAPE: 1.2571
Symbol: SAB, R^2: 0.7306, MAPE: 2.0719
Symbol: SHB, R^2: 0.9463, MAPE: 1.3359
Symbol: SSB, R^2: 0.5863, MAPE: 3.1494
Symbol: SSI, R^2: 0.9018, MAPE: 1.3897
Symbol: STB, R^2: 0.8464, MAPE: 2.4716
Symbol: TCB, R^2: 0.9644, MAPE: 1.5073
Symbol: TPB, R^2: 0.9002, MAPE: 1.7100
Symbol: VCB, R^2: 0.4702, MAPE: 1.6250
Symbol: VHM, R^2: 0.9250, MAPE: 2.1426
Symbol: VIB, R^2: 0.8590, MAPE: 1.4930
Symbol: VIC, R